## Базовый пример запуска Gamac на текстовых и визуальных данных
Было запущено через Ubuntu 22.02, cuda 12.8

### Импортируем нужные библиотеки

In [1]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd

from gamac.autoclustering import Gamac

from torchvision.datasets import CIFAR100

### Получим изначальные данные и приводим каждую модальность списком

In [2]:
# Первая загрузка будет долгая так как он грузит из open-source
cifar100 = CIFAR100('../data/cifar', download=True, train=False)

cifar_txt = [f'a photo of {cifar100.classes[img[1]]}' for img in cifar100][:100]
cifar_img = [img[0] for img in cifar100][:100]

### Инициализируем Gamac

In [11]:
# По умолчанию кол-во итераций стоит 50
gamac = Gamac(iter_limit=15)

/venv/main/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### Запустим поиск оптимального алгоритма кластеризации
Работа происходит в три этапа
1. Обработка и приведение модальностей в единый вектор
2. Определение меры - можно задать явно в target_measures, либо оставить пустым (в этом случае мера будет подобрана автоматически)
3. Поиск алгоритма кластеризации с наилучшими по выбранным мерам гиперпараметрами

In [12]:
# В данном примере мы объявили меры явно
dataset, best_model = gamac.run(text=cifar_txt, image=cifar_img, target_measures=["BR","OS","MCR","SYM"])

Detected measures: [<Internal.OS: ('os', <function os at 0x7f554078e200>)>, <Internal.MCR: ('mc_clain_rao', <function mcr at 0x7f554078c5e0>)>, <Internal.SYM: ('sym', <function sym at 0x7f554078e020>)>, <Internal.BR: ('banfield_raftery', <function br at 0x7f554078df80>)>]
=== MEASURES 0.013181924819946289s, {'OS': -367.44762141899946, 'MCR': -2.9865463579999654, 'SYM': 0.004342393636355336, 'BR': -0.4075184950847202} ===
=== ALGO 0.15881609916687012s, FailedRun, {'name': 'BisectingKMeans', 'n_clusters': 10, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 0.45218729972839355s, FailedRun, {'bandwidth': 0.7931523895473249, 'max_iter': 55, 'tol': 5.950107143772457e-05, 'centroids': array([[ 0.00410073,  0.01038773,  0.02978667, ...,  0.01159698,
        -0.00250564, -0.02343895],
       [ 0.02697312,  0.00377298,  0.0229792 , ...,  0.05553127,
         0.01284417, -0.014399  ],
       [-0.00966071,  0.02318301,  0.01761101, ..., -0.00318574,
        -0.00146954, -0.01996659]

### На выходе получаем итоговый датасет после обработки

In [13]:
print(dataset.shape)
print(dataset)

(100, 1536)
[[-0.01632632  0.02296646  0.02307877 ... -0.00527683  0.01964698
  -0.01977577]
 [ 0.00913448  0.02250634  0.03444556 ...  0.02861088 -0.00721718
  -0.04024488]
 [-0.01301966 -0.00192939  0.00460435 ... -0.00725974 -0.01104933
  -0.00826025]
 ...
 [ 0.04015086 -0.00227058  0.02453011 ...  0.02328112 -0.01052169
  -0.01894136]
 [-0.04441148  0.04697381  0.02639047 ... -0.01250509 -0.02421415
  -0.00955869]
 [-0.00078749  0.02142651  0.03256414 ...  0.00246474 -0.00090731
  -0.02039285]]


### Также получаем информацию о лучшей модели - ее наилучшей конфигурацией и финального значения мер

In [9]:
print(best_model.estimation)

{<Internal.SYM: ('sym', <function sym at 0x7f5c7525a200>)>: 0.003478495310367569, <Internal.OS: ('os', <function os at 0x7f5c7525a3e0>)>: -3.2945233592595713, <Internal.BR: ('banfield_raftery', <function br at 0x7f5c7525a160>)>: -4.529576821225851, <Internal.MCR: ('mc_clain_rao', <function mcr at 0x7f5c7525a0c0>)>: -2.1993288170466125}


### Получение меток по лучшему классификатору

In [14]:
print(best_model.model.predict(dataset))

[4 4 2 4 0 4 2 2 4 4 0 2 5 4 4 2 4 2 4 4 4 4 2 2 2 4 4 4 1 4 4 4 4 2 2 2 4
 4 4 1 2 2 4 2 2 1 4 4 2 4 2 4 4 4 2 3 0 4 2 4 2 2 2 2 4 4 2 2 2 4 2 5 4 2
 3 3 2 4 2 1 0 2 4 4 2 4 2 4 4 2 1 2 2 2 4 2 2 2 1 4]
